In [2]:
!pip install kafka-python


  Obtaining dependency information for kafka-python from https://files.pythonhosted.org/packages/4a/db/694fd552295ed091e7418d02b6268ee36092d4c93211136c448fe061fe32/kafka_python-2.3.0-py2.py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 5.4 MB/s eta 0:00:00a 0:00:01


In [3]:
from kafka import KafkaProducer
producer = KafkaProducer(bootstrap_servers="kafka:9092")
print("✅ Connexion Kafka OK")


✅ Connexion Kafka OK


In [4]:
from kafka import KafkaProducer
import requests, json, time

API_URL = "https://api.open-meteo.com/v1/forecast"
LAT, LON = 52.52, 13.41
KAFKA_TOPIC = "weather_transformed"
KAFKA_BROKER = "kafka:9092"

def fetch_weather():
    params = {
        "latitude": LAT,
        "longitude": LON,
        "current_weather": "true"
    }
    r = requests.get(API_URL, params=params, timeout=10)
    r.raise_for_status()
    return r.json().get("current_weather", {})

def transform_weather(record):
    if "temperature" in record:
        record["temp_f"] = record["temperature"] * 9/5 + 32
    record["high_wind_alert"] = record.get("windspeed", 0) > 10
    return record

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

while True:
    weather = fetch_weather()
    if weather:
        producer.send(KAFKA_TOPIC, transform_weather(weather))
        producer.flush()
        print("Sent:", weather)
    time.sleep(30)


Sent: {'time': '2026-01-22T09:45', 'interval': 900, 'temperature': -3.9, 'windspeed': 6.1, 'winddirection': 87, 'is_day': 1, 'weathercode': 0, 'temp_f': 24.98, 'high_wind_alert': False}
Sent: {'time': '2026-01-22T09:45', 'interval': 900, 'temperature': -3.9, 'windspeed': 6.1, 'winddirection': 87, 'is_day': 1, 'weathercode': 0, 'temp_f': 24.98, 'high_wind_alert': False}


KeyboardInterrupt: 